# Qiskit Avanzado: Usando Primitives (Sampler y Estimator)

La antigua manera de recuperar información de Qiskit mediante vectores estáticos o backend `get_counts()` está en declive. La gran mayoría de los trabajos avanzados se realiza a través de primitivas.

En lugar de interactuar estáticamente con un transpilador, encapsulamos en **Primitives** las ideas del experimento: `Sampler` (para distribuciones estadísticas) y `Estimator` (para energía de sistemas y valores de observables).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
import numpy as np

## 1. Muestreo de Distribuciones con SamplerV2
Creamos un circuito de campanas entrelazadas para generar un estado GHZ de 3 qubits, y mediremos la probabilidad en la salida empírica.

Nota: El sampler *debe* tener operaciones de lectura `measure` para saber de qué registros sacar la estadística.

In [ ]:
qc_ghz = QuantumCircuit(3)
qc_ghz.h(0)
qc_ghz.cx(0, 1)
qc_ghz.cx(1, 2)
qc_ghz.measure_all()

print("Circuito de GHZ:")
display(qc_ghz.draw('mpl'))

# Preparamos nuestra llamada
sampler = StatevectorSampler()
job_sampler = sampler.run([qc_ghz], shots=2000)
result_sampler = job_sampler.result()

# Extraer en V2 el diccionario usando el nombre del registro ('meas')
pub_result = result_sampler[0]
counts = pub_result.data.meas.get_counts()

print(f"\nCuentas obtenidas después del Sampling: {counts}")

## 2. Predicción de Variables Físicas con EstimatorV2
Vamos a girar un circuito en función de parámetros determinísticos $\theta$. Como observables usaremos operadores de Pauli `Z` para ver cómo la rotación continua varía imperceptiblemente el valor esperado desde el horizonte.

Nota: El estimator **no necesita mediciones explícitas**, sino el estado final superpuesto puro, que será contrastado con un operador lineal autoadjunto (observable).

In [ ]:
from qiskit.circuit import Parameter

theta = Parameter('θ')
qc_param = QuantumCircuit(1)
qc_param.rx(theta, 0)

print("Circuito paramétrico:")
display(qc_param.draw('mpl'))

# Nuestro observable será medir su alienación sobre el eje Z magnético
observable_Z = SparsePauliOp(['Z'])

estimator = StatevectorEstimator()

angles = np.linspace(0, 2 * np.pi, 20)
expected_values = []

for angle in angles:
    # Pasamos el circuito, el observable, y alimentamos el parametro theta
    # Estructura del PUB (Primitive Unified Bloc) en V2: (circuito, observables, bindings de parametros)
    job_est = estimator.run([(qc_param, observable_Z, [[angle]])])
    res = job_est.result()
    
    expected_val = res[0].data.evs[0]
    expected_values.append(expected_val)
    
print("\nValores esperados proyectivos a lo largo de un ciclo magnético Z completo:")
print([round(val, 2) for val in expected_values])